In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, before_model
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)
print(llm.invoke("Say hi in five words.").content)

Hi there, hope you're well.


In [3]:
K = 3  # human turns the model sees, including the current one


@before_model
def keep_last_k_turns(state: AgentState, runtime: Runtime):
    msgs = state["messages"]
    human_idx = [i for i, m in enumerate(msgs) if m.type == "human"]
    if len(human_idx) <= K:
        return None
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *msgs[human_idx[-K] :]]}

In [4]:
# Create an agent with windowed memory that keeps only the last K human turns
agent = create_agent(
    llm,
    system_prompt="You are a helpful assistant.",
    middleware=[keep_last_k_turns],
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "window-demo"}}


def chat(text: str) -> str:
    out = agent.invoke({"messages": [HumanMessage(text)]}, config)
    return out["messages"][-1].text

In [9]:
name = input("Enter your name: ")
print(chat(f"My name is {name}. Please remember it and greet me."))
print("\n  Topic 1:\n", chat("Tell me something interesting about Mars."))
print("\n  Topic 2:\n", chat("Now explain how plants make food."))
print("\n  Topic 3:\n", chat("Now explain where we apply calculus in Data Science."))
print("\n Follow-up Response:\n", chat("Do you remember my name?"))

Hi Savvas! Nice to meet you. I’ll remember your name for this chat. If you’d like me to remember it for future conversations, you can enable memory. If you want me to forget it after this chat, just tell me to forget my name. How can I help you today, Savvas?

  Topic 1:
 One of the most striking things about Mars is Olympus Mons, the tallest volcano in the solar system. It rises about 22 kilometers (13.6 miles) high and spans roughly 600 kilometers (370 miles) across. Mars’ low gravity and lack of plate tectonics let such a massive shield volcano form. A fun tidbit: a day on Mars (a sol) is about 24 hours 39 minutes, and you’d weigh roughly 38% of your Earth weight there.

  Topic 2:
 Plants make their own food through a process called photosynthesis, mostly in the green parts of the plant (like leaves). Here’s the gist:

- What the “food” is: glucose (a type of sugar). Plants use glucose for energy and as a building block to make other important molecules.

- Where it happens: in chl

In [10]:
for m in agent.get_state(config).values["messages"]:
    print(m.type, "|", m.text[:80])

human | Now explain how plants make food.
ai | Plants make their own food through a process called photosynthesis, mostly in th
human | Now explain where we apply calculus in Data Science.
ai | Great question. Calculus shows up in data science in several core ways. Here’s a
human | Do you remember my name?
ai | I don’t know your name yet. I don’t automatically remember personal details betw
